# Algorithmic Trading

**Installing Libraries**

In [ ]:
#!pip install tabulate
#!pip install iexfinance
#!pip install datetime

#### Import packages

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.mlab as mlab
import pandas as pd
import seaborn as sns # for making plots with seaborn
color = sns.color_palette()
import sklearn.metrics as metrics
from scipy.stats import norm 
from tabulate import tabulate

from iexfinance.stocks import get_historical_data
from iexfinance.stocks import Stock
from datetime import datetime

import warnings
warnings.filterwarnings("ignore")

#### Importing the dataset

In [ ]:
stock_prices = pd.read_csv('ICICIBANK.NS.csv')

#Glimpse of Data
stock_prices.head()

#### Fixing messy column names (containing spaces) for ease of use

In [ ]:
stock_prices.columns = stock_prices.columns.str.replace(' ', '_')

In [ ]:
stock_prices.head()

#### First, let us check the number of rows (observations) and the number of columns (variables)

In [ ]:
print('The number of rows (observations) is',stock_prices.shape[0],'\n''The number of columns (variables) is',stock_prices.shape[1])

#### Lets us plot & see price trend over time for different companies

Changing format of 'Date' variable from object to Datetime

In [ ]:
stock_prices['Date'] = [pd.to_datetime(d) for d in stock_prices['Date']]

In [ ]:
plt.figure(figsize=(10,5))
plt.scatter(stock_prices['Date'], stock_prices['Adj_Close'], edgecolors='r')
#plt.plot(stock_prices.index, stock_prices['Adj_Close'])
plt.xlabel("Date")
plt.ylabel("$ price")
plt.title("DIS Stock Price 1/1/17 - 8/1/19")

# Generating technical indicators of the stock###

#### ## Implementing moving average

In [ ]:
stock_prices["average10"] = stock_prices['Adj_Close'].rolling(window=10).mean()
stock_prices["average20"] = stock_prices['Adj_Close'].rolling(window=20).mean()
stock_prices["average10"].head(20)

In [ ]:
stock_prices["std10"] = stock_prices['Adj_Close'].rolling(window=10).std()
stock_prices["std20"] = stock_prices['Adj_Close'].rolling(window=20).std()

#### Computing RSI Indicator
Relative Strength Index (RSI) - The Relative Strength Index (RSI) calculates a ratio of the recent upward price movements to the absolute price movement. The RSI ranges from 0 to 100. The RSI is interpreted as an overbought/oversold indicator when the value is over 70/below 30.

In [ ]:
def RSI (data, time_window):
    diff = data.diff(1).dropna()        # diff in one field(one day)

    #this preservers dimensions of diff values
    up_chg = 0 * diff
    down_chg = 0 * diff
    
    # up change is equal to the positive difference, otherwise equal to zero
    up_chg[diff > 0] = diff[ diff>0 ]
    
    # down change is equal to negative deifference, otherwise equal to zero
    down_chg[diff < 0] = diff[ diff < 0 ]
    
    up_chg_avg   = up_chg.ewm(com=time_window-1 , min_periods=time_window).mean()
    down_chg_avg = down_chg.ewm(com=time_window-1 , min_periods=time_window).mean()
    
    rs = abs(up_chg_avg/down_chg_avg)
    rsi = 100 - 100/(1+rs)
    return rsi

In [ ]:
stock_prices['rsi5'] = RSI(stock_prices['Adj_Close'], 5)
stock_prices['rsi14'] = RSI(stock_prices['Adj_Close'], 14)

#### Calculating MACD (Moving Average Convergence Divergence) 
MACD is the difference between two Exponential Moving Averages

In [ ]:
stock_prices['12d_EMA'] = stock_prices['Adj_Close'].ewm(span=12, adjust=False).mean()
stock_prices['26d_EMA'] = stock_prices['Adj_Close'].ewm(span=26, adjust=False).mean()

stock_prices[['Adj_Close','12d_EMA','26d_EMA']].plot(figsize=(10,5))
plt.show()

#### Calculate the difference between 26 day & 12 day Moving averages

In [ ]:
stock_prices['macd'] = stock_prices['12d_EMA'] - stock_prices['26d_EMA'] 

#### Calculate Signal

In [ ]:
stock_prices['macdsignal'] = stock_prices.macd.ewm(span=9, adjust=False).mean()

stock_prices[['macd','macdsignal']].plot(figsize=(10,5))
plt.show()

#### Compute the Bollinger Bands using the 20-day Moving average

In [ ]:
MA = stock_prices['Adj_Close'].rolling(window=20).mean()
SD = stock_prices['Adj_Close'].rolling(window=20).std()
stock_prices['UpperBB'] = MA + (2 * SD) 
stock_prices['LowerBB'] = MA - (2 * SD)

## Steps: We clean the output and generated a column called direction basis the following:
- If Price > upper Bollinger band, and MACD value > MACD signal -> Buy signal (1)
- If Price < lower Bollinger band, and MACD value < MACD signal -> Sell signal (-1)
- Else, Out of the market -> Signal OOM(0)

In [ ]:
# Define Signal
stock_prices['Direction'] = np.where((stock_prices['macd'] > stock_prices['macdsignal']) & 
                                      (stock_prices['Adj_Close'] > stock_prices['UpperBB']), 1, 
                                     np.where((stock_prices['macd'] < stock_prices['macdsignal']) & 
                                               (stock_prices['Adj_Close'] < stock_prices['LowerBB']),-1,0))



stock_prices['Direction']

#### Checking distribution of response variable

In [ ]:
stock_prices['Direction'].value_counts()

#### Create the plot

In [ ]:
pd.concat([stock_prices.Adj_Close,stock_prices.UpperBB,stock_prices.LowerBB],axis=1).plot(figsize=(9,5),grid=True)

In [ ]:
ICICI_final = stock_prices[['Adj_Close','average10', 'average20', 'std10', 'std20', 'rsi5', 'rsi14', 'Direction']]

#### Creating training & test datasets for model building

In [ ]:
X = ICICI_final.drop(['Direction'], axis=1)
y = ICICI_final['Direction']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3,random_state=142,stratify=ICICI_final['Direction'])
                                             
Train_icici = pd.concat([X_train,y_train], axis=1)
Test_icici = pd.concat([X_test,y_test], axis=1)

#Exporting training & test dataset
Train_icici.to_csv('Train_icici.csv',index=False)
Test_icici.to_csv('Test_icici.csv',index=False)

#### Importing training & test datasets

In [ ]:
Train_icici = pd.read_csv('Train_icici.csv')
Test_icici = pd.read_csv('Test_icici.csv')

#### Checking frequency of response variables

In [ ]:
ICICI_final['Direction'].value_counts()

In [ ]:
Train_icici['Direction'].value_counts()

In [ ]:
Test_icici['Direction'].value_counts()

# Model building:
We choose to model this data using Decision Trees such as CART & RandomForest. 

Decision Trees are simple yet powerful and require minimal data preparation such as scaling and missing value treatment etc.

Here, Before we proceed for model building, lets do missing value imputation (Scaling is not required while modelling trees)

In [ ]:
#creating list of columns in the dataset
col_train=list(Train_icici)
col_test=list(Test_icici)

from sklearn.impute import SimpleImputer
imputer = SimpleImputer(missing_values=np.nan, strategy='median')

Train_icici = pd.DataFrame(imputer.fit_transform(Train_icici))
Test_icici = pd.DataFrame(imputer.fit_transform(Test_icici))

Train_icici.columns=col_train
Test_icici.columns=col_test

Train_icici.head()

In [ ]:
X_train = Train_icici.drop(['Direction'], axis=1)
y_train = Train_icici['Direction']

X_test = Test_icici.drop(['Direction'], axis=1)
y_test = Test_icici['Direction']

## Import CART model

In [ ]:
#Visualising Dtree in Python
from sklearn import tree
import matplotlib.pyplot as plt
clf = tree.DecisionTreeClassifier(max_leaf_nodes = 5)
plt.figure(figsize=(12,7))
tree.plot_tree(clf.fit(X_train, y_train),filled=True)
plt.show()

#### Fit CART model

In [ ]:
clf = clf.fit(X_train, y_train)

#### Predict on training & test datasets

In [ ]:
y_pred_train = clf.predict(X_train)
y_pred_test = clf.predict(X_test)

model_score_train = clf.score(X_train, y_train)
model_score_test = clf.score(X_test, y_test)

##### Confusion matrix report

In [ ]:
print(metrics.classification_report(y_train,y_pred_train,digits=3))
print(metrics.classification_report(y_test,y_pred_test,digits=3))

***Observation***: While model is able to identify the Buy and OOM signals, it is not able to identify the Sell signals correctly

### Import Random Forest Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
#Create a Gaussian Classifier
clfRF=RandomForestClassifier(n_estimators=100, max_leaf_nodes = 5)

#### Fit RF model

In [ ]:
clfRF.fit(X_train,y_train)

#### Predict on training & test datasets

In [ ]:
y_pred_train = clfRF.predict(X_train)
y_pred_test = clfRF.predict(X_test)

RFmodel_score_train = clfRF.score(X_train, y_train)
RFmodel_score_test = clfRF.score(X_test, y_test)

#### Classification report

In [ ]:
print(metrics.classification_report(y_train,y_pred_train,digits=3))
print(metrics.classification_report(y_test,y_pred_test,digits=3))

***Observation***: While model is able to identify the Buy and OOM signals, it is not able to identify the Sell signals correctly

Both the models: Decision tree (CART) & RandomForest can be further improved using hyperparameter tuning or using advanced algorithm such as Boosting techniques

# End